In [37]:
learning_rate = 1e-4
batch_size = 128 # Minibatch size
epochs = 100

# Optimizer parameters
beta_momentum = 0.9 # Adam 1st moment decay (beta1)
beta_rmsprop = 0.999 # Adam 2nd moment decay (beta2)
epsilon = 1e-8 # small value to avoid division by zero

# Regularization parameters
weight_decay = 1e-4 # AdamW decoupled weight decay
keep_prob = 0.8 # Dropout keep probability
label_smoothing = 0.1 # Label smoothing epsilon (0 = off)

bn_momentum = 0.9 # Batch normalization momentum
bn_epsilon = 1e-5 # Batch normalization epsilon


In [38]:
import numpy as np
BASE_PATH = '../../assets/raw'

def load_image(file):
    with open(file, 'rb') as f:
        f.seek(16)  # Skip the header
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(-1, 28, 28)  # Reshape to (num_images, height, width)

def load_label(file):
    with open(file, 'rb') as f:
        f.seek(8)  # Skip the header
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data


In [39]:
def one_hot_encode(Y, num_classes):
    return np.eye(num_classes)[Y]

In [40]:
def split_mini_batches(X, Y, batch_size):
    m = X.shape[1]
    perm = np.random.permutation(m)
    X, Y_shuf = X[:, perm], Y[:, perm]
    return [(X[:, i:i + batch_size], Y_shuf[:, i:i + batch_size])
               for i in range(0, m, batch_size)]

In [41]:
def relu(Z):
    return np.maximum(0, Z)

In [42]:
def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True)) # sub max for numerical stability
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

In [43]:
def relu_derivative(Z):
    return (Z > 0).astype(float) # derivative of ReLU (1 if Z > 0 else 0)

In [44]:
def forward_propagation(X, L, parameters, bn_parameters, training=True):
    cache = {}
    A = X # A0 (input layer)

    for layer in range(1, L + 1):
        W = parameters[f'W{layer}']
        b = parameters[f'b{layer}']

        Z = np.dot(W, A) + b # Z = W * A + b

        if layer < L:
            gamma = bn_parameters[f'gamma{layer}']
            beta  = bn_parameters[f'beta{layer}']

            if training:
                mu  = np.mean(Z, axis=1, keepdims=True)
                var = np.var(Z,  axis=1, keepdims=True)
                # update running stats (used only at inference)
                bn_parameters[f'mu{layer}']  = bn_momentum * bn_parameters[f'mu{layer}']  + (1 - bn_momentum) * mu
                bn_parameters[f'var{layer}'] = bn_momentum * bn_parameters[f'var{layer}'] + (1 - bn_momentum) * var
            else:
                mu  = bn_parameters[f'mu{layer}']
                var = bn_parameters[f'var{layer}']

            Z_hat   = (Z - mu) / np.sqrt(var + bn_epsilon)
            Z_tilde = gamma * Z_hat + beta

            A = relu(Z_tilde)

            if training and keep_prob < 1.0:
                D = (np.random.rand(*A.shape) < keep_prob).astype(A.dtype)
                A = A * D / keep_prob
                cache[f'D{layer}'] = D

            # cache everything bn_backward needs (NOT in bn_parameters — those are running stats)
            cache[f'Z{layer}']       = Z
            cache[f'mu{layer}']      = mu
            cache[f'var{layer}']     = var
            cache[f'Z_hat{layer}']   = Z_hat
            cache[f'Z_tilde{layer}'] = Z_tilde
            cache[f'A{layer}']       = A
        else:
            A = softmax(Z) # output layer uses softmax
            cache[f'Z{layer}'], cache[f'A{layer}'] = Z, A

    return A, cache


In [45]:
def loss(Y, Y_hat):
    m = Y.shape[1]
    return -np.sum(Y * np.log(np.clip(Y_hat, 1e-12, 1.0))) / m # Cross-entropy loss

In [46]:
def accuracy(Y, Y_hat):
    predictions = np.argmax(Y_hat, axis=0)
    labels = np.argmax(Y, axis=0)
    return np.mean(predictions == labels) * 100

In [47]:
def bn_backward(dZ_tilde, cache, layer, gamma, bn_eps=1e-5):
    Z      = cache[f'Z{layer}']
    mu     = cache[f'mu{layer}']
    var    = cache[f'var{layer}']
    Z_hat  = cache[f'Z_hat{layer}']
    m      = Z.shape[1]
    inv_std = 1.0 / np.sqrt(var + bn_eps)

    dgamma = np.sum(dZ_tilde * Z_hat, axis=1, keepdims=True)
    dbeta  = np.sum(dZ_tilde,         axis=1, keepdims=True)

    dZ_hat = dZ_tilde * gamma
    # the compact form of dZ (derived from chain rule through mean & var)
    dZ = (1.0 / m) * inv_std * (
        m * dZ_hat
        - np.sum(dZ_hat, axis=1, keepdims=True)
        - Z_hat * np.sum(dZ_hat * Z_hat, axis=1, keepdims=True)
    )
    return dZ, dgamma, dbeta

In [48]:
def backward_propagation(X, dZ, m, L, parameters, bn_parameters, cache, training=True):
    grads = {}

    for layer in reversed(range(1, L + 1)):
        A_prev = cache[f'A{layer - 1}'] if layer > 1 else X
        W = parameters[f'W{layer}']

        grads[f'dW{layer}'] = (1 / m) * np.dot(dZ, A_prev.T)
        grads[f'db{layer}'] = (1 / m) * np.sum(dZ, axis=1, keepdims=True)

        if layer > 1:
            dA_prev = np.dot(W.T, dZ)

            # apply dropout mask used in forward propagation
            if training and keep_prob < 1.0 and f'D{layer - 1}' in cache:
                D = cache[f'D{layer - 1}']
                dA_prev = dA_prev * D / keep_prob

            # back through ReLU (applied on Z_tilde, not Z)
            Z_tilde_prev = cache[f'Z_tilde{layer - 1}']
            dZ_tilde_prev = dA_prev * relu_derivative(Z_tilde_prev)

            # back through BN
            gamma_prev = bn_parameters[f'gamma{layer - 1}']
            dZ, dgamma, dbeta = bn_backward(dZ_tilde_prev, cache, layer - 1, gamma_prev, bn_epsilon)
            grads[f'dgamma{layer - 1}'] = dgamma
            grads[f'dbeta{layer - 1}']  = dbeta

    return grads


### Initialization

In [49]:
X_train = load_image(f'{BASE_PATH}/train-images-idx3-ubyte').reshape(-1, 28*28).T  # Reshape to (num_features, num_samples)
Y_train = one_hot_encode(load_label(f'{BASE_PATH}/train-labels-idx1-ubyte'), num_classes=10).T # One-hot encode (1, num_samples) -> (num_classes, num_samples)

X_dev = load_image(f'{BASE_PATH}/t10k-images-idx3-ubyte').reshape(-1, 28*28).T  # Reshape to (num_features, num_samples)
Y_dev = one_hot_encode(load_label(f'{BASE_PATH}/t10k-labels-idx1-ubyte'), num_classes=10).T # One-hot encode (1, num_samples) -> (num_classes, num_samples)

In [50]:
X_train = X_train / 255.0  # Normalize pixel values to [0, 1]
X_dev = X_dev / 255.0 

In [51]:
units = [X_train.shape[0], 256, 128, Y_train.shape[0]] # input, hidden, output layers

parameters = {}
for layer in range(1, len(units)):
    # W: (layer units, layer-1 units)
    parameters[f'W{layer}'] = np.random.randn(units[layer], units[layer - 1]) * np.sqrt(2 / units[layer - 1]) # He init
    parameters[f'b{layer}'] = np.zeros((units[layer], 1))

bn_parameters = {}
for layer in range(1, len(units) - 1):  # no BN on the output layer
    bn_parameters[f'gamma{layer}'] = np.ones((units[layer], 1))
    bn_parameters[f'beta{layer}']  = np.zeros((units[layer], 1))
    bn_parameters[f'mu{layer}']    = np.zeros((units[layer], 1))  # running
    bn_parameters[f'var{layer}']   = np.ones((units[layer], 1))   # running

adam_parameters = {}
for layer in range(1, len(units)):
    adam_parameters[f'VdW{layer}'] = np.zeros_like(parameters[f'W{layer}'])
    adam_parameters[f'Vdb{layer}'] = np.zeros_like(parameters[f'b{layer}'])
    adam_parameters[f'SdW{layer}'] = np.zeros_like(parameters[f'W{layer}'])
    adam_parameters[f'Sdb{layer}'] = np.zeros_like(parameters[f'b{layer}'])
    if layer < len(units) - 1:  # Adam moments for BN params
        adam_parameters[f'Vdgamma{layer}'] = np.zeros_like(bn_parameters[f'gamma{layer}'])
        adam_parameters[f'Sdgamma{layer}'] = np.zeros_like(bn_parameters[f'gamma{layer}'])
        adam_parameters[f'Vdbeta{layer}']  = np.zeros_like(bn_parameters[f'beta{layer}'])
        adam_parameters[f'Sdbeta{layer}']  = np.zeros_like(bn_parameters[f'beta{layer}'])


### Model

In [52]:
L = len(units) - 1 # number of layers (excluding input layer)
t = 0 # timestep for Adam optimizer

best_dev_cost = float('inf')
best_params = None
best_bn_params = None
wait = 0
patience = 20

for epoch in range(epochs):
    batches = split_mini_batches(X_train, Y_train, batch_size)

    epoch_cost = 0
    for X_batch, Y_batch in batches:
        m = X_batch.shape[1]
        t += 1

        # forward prop (training mode)
        A_L, cache = forward_propagation(X_batch, L, parameters, bn_parameters, training=True)

        # label smoothing: soften targets for the gradient only
        if label_smoothing > 0:
            K = Y_batch.shape[0]
            Y_target = Y_batch * (1 - label_smoothing) + label_smoothing / K
        else:
            Y_target = Y_batch

        # loss (reported on hard labels so it's comparable to prior runs)
        cost = loss(Y_batch, A_L)
        epoch_cost += cost * m

        # back prop (uses smoothed targets)
        dZ_L = A_L - Y_target
        grads = backward_propagation(X_batch, dZ_L, m, L, parameters, bn_parameters, cache, training=True)

        # update parameters
        for layer in range(1, L + 1):
            adam_parameters[f'VdW{layer}'] = beta_momentum * adam_parameters[f'VdW{layer}'] + (1 - beta_momentum) * grads[f'dW{layer}']
            adam_parameters[f'Vdb{layer}'] = beta_momentum * adam_parameters[f'Vdb{layer}'] + (1 - beta_momentum) * grads[f'db{layer}']
            adam_parameters[f'SdW{layer}'] = beta_rmsprop * adam_parameters[f'SdW{layer}'] + (1 - beta_rmsprop) * (grads[f'dW{layer}'] ** 2)
            adam_parameters[f'Sdb{layer}'] = beta_rmsprop * adam_parameters[f'Sdb{layer}'] + (1 - beta_rmsprop) * (grads[f'db{layer}'] ** 2)

            VdW_c = adam_parameters[f'VdW{layer}'] / (1 - beta_momentum ** t)
            Vdb_c = adam_parameters[f'Vdb{layer}'] / (1 - beta_momentum ** t)
            SdW_c = adam_parameters[f'SdW{layer}'] / (1 - beta_rmsprop ** t)
            Sdb_c = adam_parameters[f'Sdb{layer}'] / (1 - beta_rmsprop ** t)

            parameters[f'W{layer}'] -= learning_rate * (
                VdW_c / (np.sqrt(SdW_c) + epsilon) + weight_decay * parameters[f'W{layer}']
            )
            parameters[f'b{layer}'] -= learning_rate * (Vdb_c / (np.sqrt(Sdb_c) + epsilon))

            if layer < L:
                for name in ('gamma', 'beta'):
                    g = grads[f'd{name}{layer}']
                    adam_parameters[f'Vd{name}{layer}'] = beta_momentum * adam_parameters[f'Vd{name}{layer}'] + (1 - beta_momentum) * g
                    adam_parameters[f'Sd{name}{layer}'] = beta_rmsprop * adam_parameters[f'Sd{name}{layer}'] + (1 - beta_rmsprop) * (g ** 2)
                    V_c = adam_parameters[f'Vd{name}{layer}'] / (1 - beta_momentum ** t)
                    S_c = adam_parameters[f'Sd{name}{layer}'] / (1 - beta_rmsprop ** t)
                    bn_parameters[f'{name}{layer}'] -= learning_rate * V_c / (np.sqrt(S_c) + epsilon)

    epoch_cost /= X_train.shape[1]

    # evaluate on dev set EVERY epoch (needed for early stopping)
    A_dev,   _ = forward_propagation(X_dev,   L, parameters, bn_parameters, training=False)
    dev_cost   = loss(Y_dev, A_dev)
    dev_acc    = accuracy(Y_dev, A_dev)

    if epoch % 10 == 0 or epoch == epochs - 1:
        A_train, _ = forward_propagation(X_train, L, parameters, bn_parameters, training=False)
        train_acc  = accuracy(Y_train, A_train)
        print(f'Epoch {epoch + 1}/{epochs}, Training Loss: {epoch_cost:.4f}, Dev Loss: {dev_cost:.4f}, Train Acc: {train_acc:.2f}%, Dev Acc: {dev_acc:.2f}%')

    # early stopping
    if dev_cost < best_dev_cost:
        best_dev_cost  = dev_cost
        best_params    = {k: v.copy() for k, v in parameters.items()}
        best_bn_params = {k: v.copy() for k, v in bn_parameters.items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch + 1}. Best Dev Loss: {best_dev_cost:.4f}")
            break

# restore best weights
if best_params is not None:
    parameters    = best_params
    bn_parameters = best_bn_params


Epoch 1/100, Training Loss: 0.9024, Dev Loss: 0.5709, Train Acc: 83.38%, Dev Acc: 81.87%
Epoch 11/100, Training Loss: 0.4249, Dev Loss: 0.4105, Train Acc: 90.09%, Dev Acc: 87.36%
Epoch 21/100, Training Loss: 0.3730, Dev Loss: 0.3877, Train Acc: 91.97%, Dev Acc: 88.45%
Epoch 31/100, Training Loss: 0.3376, Dev Loss: 0.3700, Train Acc: 93.40%, Dev Acc: 88.97%
Epoch 41/100, Training Loss: 0.3118, Dev Loss: 0.3641, Train Acc: 94.41%, Dev Acc: 89.36%
Epoch 51/100, Training Loss: 0.2881, Dev Loss: 0.3581, Train Acc: 95.35%, Dev Acc: 89.48%
Epoch 61/100, Training Loss: 0.2714, Dev Loss: 0.3598, Train Acc: 96.00%, Dev Acc: 89.32%
Epoch 71/100, Training Loss: 0.2581, Dev Loss: 0.3570, Train Acc: 96.95%, Dev Acc: 89.88%
Epoch 81/100, Training Loss: 0.2406, Dev Loss: 0.3556, Train Acc: 97.42%, Dev Acc: 89.94%
Epoch 91/100, Training Loss: 0.2343, Dev Loss: 0.3655, Train Acc: 97.32%, Dev Acc: 89.47%
Epoch 100/100, Training Loss: 0.2240, Dev Loss: 0.3576, Train Acc: 98.03%, Dev Acc: 89.65%
